In [0]:
# ─────────────────────────────────────────────────────────────
# SILVER — dim_match   (grain: 1 row per match; needs only bronze)
# Pattern: VARIANT.info  ->  explicit StructType  ->  typed columns
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               BooleanType, ArrayType)

CATALOG = "cricket"
BRONZE_TABLE = f"{CATALOG}.bronze.raw_matches"
SILVER_DIM_MATCH = f"{CATALOG}.silver.dim_match"

# --- explicit schema for the info subtree (only what dim_match needs) ---
event_schema = StructType([
    StructField("name",         StringType()),
    StructField("match_number", IntegerType()),
    StructField("group",        StringType()),   # can be int OR string in source -> string
    StructField("stage",        StringType()),
])

outcome_by_schema = StructType([
    StructField("runs",    IntegerType()),
    StructField("wickets", IntegerType()),
    StructField("innings", IntegerType()),       # present (=1) when won by an innings
])

outcome_schema = StructType([
    StructField("winner",     StringType()),
    StructField("result",     StringType()),     # tie / draw / no result
    StructField("method",     StringType()),     # D/L, VJD, ...
    StructField("eliminator", StringType()),     # super-over winner
    StructField("bowl_out",   StringType()),      # bowl-out winner
    StructField("by",         outcome_by_schema),
])

toss_schema = StructType([
    StructField("winner",      StringType()),
    StructField("decision",    StringType()),
    StructField("uncontested", BooleanType()),
])

info_schema = StructType([
    StructField("match_type",        StringType()),
    StructField("match_type_number", IntegerType()),
    StructField("balls_per_over",    IntegerType()),
    StructField("gender",            StringType()),
    StructField("team_type",         StringType()),
    StructField("season",            StringType()),
    StructField("dates",             ArrayType(StringType())),
    StructField("teams",             ArrayType(StringType())),
    StructField("venue",             StringType()),
    StructField("city",              StringType()),
    StructField("overs",             IntegerType()),   # scheduled overs (null for Tests)
    StructField("event",             event_schema),
    StructField("toss",              toss_schema),
    StructField("outcome",           outcome_schema),
])

In [0]:
# ─────────────────────────────────────────────────────────────
# Parse + shape dim_match
# ─────────────────────────────────────────────────────────────
bronze = spark.table(BRONZE_TABLE)

info = (bronze
    .withColumn("info", F.from_json(F.expr("to_json(data:info)"), info_schema))
    .withColumn("meta_data_version", F.col("data_version"))   # already typed in bronze
)

dim_match = (info.select(
    F.col("match_id"),
    F.col("info.match_type").alias("match_type"),
    # match_format: statistical bucket. The Hundred (T20 + 5 balls) folds into T20.
    # IT20 / ODM / MDM stay distinct — they carry the domestic/international signal.
    F.when((F.col("info.match_type") == "T20") & (F.col("info.balls_per_over") == 5),
           F.lit("T20"))
     .otherwise(F.col("info.match_type")).alias("match_format"),
    # competition_variant: keep The Hundred visible for the opt-in split
    F.when((F.col("info.match_type") == "T20") & (F.col("info.balls_per_over") == 5),
           F.lit("The Hundred"))
     .otherwise(F.lit(None).cast("string")).alias("competition_variant"),
    # is_international: one-column tier filter (intl vs domestic/league)
    (F.col("info.team_type") == "international").alias("is_international"),
    F.col("info.season").alias("season"),
    # season_start_year: "2020/21" -> 2020, "2026" -> 2026
    F.regexp_extract(F.col("info.season"), r"^(\d{4})", 1).cast("int").alias("season_start_year"),
    F.col("info.gender").alias("gender"),
    F.col("info.event.name").alias("event_name"),
    F.col("info.event.match_number").alias("event_match_number"),
    F.col("info.event.group").alias("event_group"),
    F.col("info.event.stage").alias("event_stage"),
    F.col("info.teams")[0].alias("team_a"),
    F.col("info.teams")[1].alias("team_b"),
    F.col("info.venue").alias("venue"),
    F.col("info.city").alias("city"),
    F.to_date(F.array_min(F.col("info.dates"))).alias("start_date"),
    F.to_date(F.array_max(F.col("info.dates"))).alias("end_date"),
    F.col("info.balls_per_over").alias("balls_per_over"),
    F.col("info.overs").alias("scheduled_overs"),
    F.col("info.toss.winner").alias("toss_winner"),
    F.col("info.toss.decision").alias("toss_decision"),
    F.col("info.toss.uncontested").alias("toss_uncontested"),
    F.col("info.outcome.winner").alias("outcome_winner"),
    F.col("info.outcome.result").alias("outcome_result"),
    F.col("info.outcome.by.runs").alias("outcome_by_runs"),
    F.col("info.outcome.by.wickets").alias("outcome_by_wickets"),
    F.col("info.outcome.by.innings").isNotNull().alias("outcome_by_innings"),
    F.col("info.outcome.method").alias("outcome_method"),
    F.col("info.outcome.eliminator").alias("outcome_eliminator"),
    F.col("info.outcome.bowl_out").alias("outcome_bowl_out"),
    # won_by_team: coalesce(winner, eliminator, bowl_out) — one clean "who won"
    F.coalesce(F.col("info.outcome.winner"),
               F.col("info.outcome.eliminator"),
               F.col("info.outcome.bowl_out")).alias("won_by_team"),
    F.col("data_version").alias("data_version"),
    F.col("revision").alias("revision"),
))

In [0]:
# ─────────────────────────────────────────────────────────────
# Write — incremental replace on the changed match_ids
# (idempotent: rebuilds only the matches present in this batch)
# ─────────────────────────────────────────────────────────────
from delta.tables import DeltaTable

if not spark.catalog.tableExists(SILVER_DIM_MATCH):
    (dim_match.write.format("delta").clusterBy("match_id")
        .saveAsTable(SILVER_DIM_MATCH))
    print("Created", SILVER_DIM_MATCH)
else:
    tgt = DeltaTable.forName(spark, SILVER_DIM_MATCH)
    (tgt.alias("t").merge(dim_match.alias("s"), "t.match_id = s.match_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print("Merged", SILVER_DIM_MATCH)

# verify
spark.sql(f"""
  SELECT match_format, competition_variant, is_international, count(*) matches
  FROM {CATALOG}.silver.dim_match
  GROUP BY ALL ORDER BY matches DESC
""").show()

In [0]:
# ─────────────────────────────────────────────────────────────
# SILVER — registry, dim_player, player_name  (from bronze VARIANT)
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType

CATALOG = "cricket"
BRONZE_TABLE      = f"{CATALOG}.bronze.raw_matches"
SILVER_DELIVERIES = f"{CATALOG}.silver.deliveries"
SILVER_REGISTRY   = f"{CATALOG}.silver.match_registry"   # (match_id, name, person_id)
SILVER_DIM_PLAYER = f"{CATALOG}.silver.dim_player"
SILVER_PLAYER_NAME= f"{CATALOG}.silver.player_name"

bronze = spark.table(BRONZE_TABLE)

# registry.people is a name->id MAP; explode to (match_id, name, person_id)
registry = (bronze
    .withColumn("people",
                F.from_json(F.expr("to_json(data:info:registry:people)"),
                            MapType(StringType(), StringType())))
    .select("match_id", F.explode("people").alias("name", "person_id"))
)
registry.write.format("delta").mode("overwrite").clusterBy("match_id").saveAsTable(SILVER_REGISTRY)

In [0]:
# ─────────────────────────────────────────────────────────────
# SILVER — deliveries  (grain: 1 row per delivery entry)
# innings -> overs -> deliveries, posexplode to keep positions
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               BooleanType, ArrayType, DoubleType)

CATALOG = "cricket"
BRONZE_TABLE       = f"{CATALOG}.bronze.raw_matches"
SILVER_DELIVERIES  = f"{CATALOG}.silver.deliveries"

# --- schemas for the innings subtree ---
runs_schema = StructType([
    StructField("batter",       IntegerType()),
    StructField("extras",       IntegerType()),
    StructField("total",        IntegerType()),
    StructField("non_boundary", BooleanType()),
])
extras_schema = StructType([
    StructField("wides",    IntegerType()),
    StructField("noballs",  IntegerType()),
    StructField("byes",     IntegerType()),
    StructField("legbyes",  IntegerType()),
    StructField("penalty",  IntegerType()),
])
fielder_schema = StructType([
    StructField("name",       StringType()),
    StructField("substitute", BooleanType()),
])
wicket_schema = StructType([
    StructField("player_out", StringType()),
    StructField("kind",       StringType()),
    StructField("fielders",   ArrayType(fielder_schema)),
])
delivery_schema = StructType([
    StructField("actual_delivery", StringType()),
    StructField("batter",          StringType()),
    StructField("bowler",          StringType()),
    StructField("non_striker",     StringType()),
    StructField("runs",            runs_schema),
    StructField("extras",          extras_schema),
    StructField("wickets",         ArrayType(wicket_schema)),
])
over_schema = StructType([
    StructField("over",        IntegerType()),
    StructField("deliveries",  ArrayType(delivery_schema)),
])
innings_schema = ArrayType(StructType([
    StructField("team",       StringType()),
    StructField("super_over", BooleanType()),
    StructField("overs",      ArrayType(over_schema)),
]))

In [0]:
# ─────────────────────────────────────────────────────────────
# Parse + three-level explode
# ─────────────────────────────────────────────────────────────
bronze = spark.table(BRONZE_TABLE)

# teams array (to derive bowling_team = the other team)
parsed = (bronze
    .withColumn("innings", F.from_json(F.expr("to_json(data:innings)"), innings_schema))
    .withColumn("teams", F.from_json(F.expr("to_json(data:info:teams)"), ArrayType(StringType())))
    .select("match_id", "revision", "teams", "innings")
)

# Level 1: innings  (posexplode -> innings_number). explode_outer so a
# match never vanishes; forfeited innings (no overs) survive as nulls.
lvl_inn = (parsed
    .select("match_id", "revision", "teams",
            F.posexplode_outer("innings").alias("inn_idx", "inn"))
    .withColumn("innings_number", F.col("inn_idx") + 1)
    .withColumn("batting_team", F.col("inn.team"))
    .withColumn("is_super_over", F.coalesce(F.col("inn.super_over"), F.lit(False)))
    # bowling_team = the element of teams that isn't batting_team
    .withColumn("bowling_team", F.element_at(F.array_except(F.col("teams"), F.array(F.col("inn.team"))), 1))
)

# Level 2: overs
lvl_over = (lvl_inn
    .select("match_id", "revision", "innings_number", "batting_team",
            "bowling_team", "is_super_over",
            F.explode_outer("inn.overs").alias("over"))
    .withColumn("over_number", F.col("over.over"))
)

# Level 3: deliveries (posexplode -> ball_seq within the over)
lvl_ball = (lvl_over
    .select("match_id", "revision", "innings_number", "batting_team",
            "bowling_team", "is_super_over", "over_number",
            F.posexplode_outer("over.deliveries").alias("ball_idx", "d"))
    .withColumn("ball_seq", F.col("ball_idx") + 1)
    .filter(F.col("d").isNotNull())   # drop the null rows left by forfeited/empty overs
)

In [0]:
# ─── deliveries: flatten -> resolve ids -> (write) — one canonical cell ───

# 1) FLATTEN: build deliveries from the exploded lvl_ball
deliveries = (lvl_ball.select(
    "match_id", "innings_number", "is_super_over",
    "batting_team", "bowling_team", "over_number", "ball_seq",
    F.col("d.actual_delivery").alias("actual_delivery"),
    F.col("d.batter").alias("batter"),
    F.col("d.bowler").alias("bowler"),
    F.col("d.non_striker").alias("non_striker"),
    F.col("d.runs.batter").alias("runs_batter"),
    F.col("d.runs.extras").alias("runs_extras"),
    F.col("d.runs.total").alias("runs_total"),
    F.col("d.runs.non_boundary").alias("non_boundary"),
    F.coalesce(F.col("d.extras.wides"),   F.lit(0)).alias("extra_wides"),
    F.coalesce(F.col("d.extras.noballs"), F.lit(0)).alias("extra_noballs"),
    F.coalesce(F.col("d.extras.byes"),    F.lit(0)).alias("extra_byes"),
    F.coalesce(F.col("d.extras.legbyes"), F.lit(0)).alias("extra_legbyes"),
    F.coalesce(F.col("d.extras.penalty"), F.lit(0)).alias("extra_penalty"),
    (F.size(F.coalesce(F.col("d.wickets"), F.array())) > 0).alias("is_wicket"),
    F.size(F.coalesce(F.col("d.wickets"), F.array())).alias("wicket_count"),
    F.col("d.wickets")[0]["kind"].alias("dismissal_kind"),
    F.col("d.wickets")[0]["player_out"].alias("player_out"),
    F.col("revision"),
))

# 2) RESOLVE ids against each match's registry (join on match_id + name)
reg = spark.table(f"{CATALOG}.silver.match_registry")

def resolve(df, name_col, id_col):
    r = reg.select("match_id",
                   F.col("name").alias(name_col),
                   F.col("person_id").alias(id_col))
    return df.join(r, ["match_id", name_col], "left")

deliveries = resolve(deliveries, "batter",      "batter_id")
deliveries = resolve(deliveries, "bowler",      "bowler_id")
deliveries = resolve(deliveries, "non_striker", "non_striker_id")
deliveries = resolve(deliveries, "player_out",  "player_out_id")

In [0]:
# ─────────────────────────────────────────────────────────────
# Write — incremental replace on the batch's matches
# ─────────────────────────────────────────────────────────────
from delta.tables import DeltaTable

if not spark.catalog.tableExists(SILVER_DELIVERIES):
    (deliveries.write.format("delta").clusterBy("match_id")
        .saveAsTable(SILVER_DELIVERIES))
    print("Created", SILVER_DELIVERIES)
else:
    # replace all rows for the matches in this batch (a revised match fully restates)
    match_ids = [r.match_id for r in deliveries.select("match_id").distinct().collect()]
    tgt = DeltaTable.forName(spark, SILVER_DELIVERIES)
    (tgt.alias("t").merge(
        deliveries.alias("s"),
        "t.match_id = s.match_id AND t.innings_number = s.innings_number "
        "AND t.over_number = s.over_number AND t.ball_seq = s.ball_seq")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .whenNotMatchedBySourceDelete(
        condition=F.col("t.match_id").isin(match_ids))   # drop stale balls of revised matches
     .execute())
    print("Merged", SILVER_DELIVERIES)

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify — the ball-level truth checks
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
  SELECT
    (SELECT count(*) FROM {SILVER_DELIVERIES})                              AS total_balls,
    (SELECT count(DISTINCT match_id) FROM {SILVER_DELIVERIES})              AS matches,
    (SELECT count(*) FROM {SILVER_DELIVERIES} WHERE is_super_over)          AS super_over_balls,
    (SELECT max(wicket_count) FROM {SILVER_DELIVERIES})                     AS max_wickets_on_a_ball
""").show()

# runs sanity: total = batter + extras, on every ball
spark.sql(f"""
  SELECT count(*) AS runs_mismatch
  FROM {SILVER_DELIVERIES}
  WHERE runs_total <> runs_batter + runs_extras
""").show()

In [0]:
# ─────────────────────────────────────────────────────────────
# Participant ids = anyone who appears in deliveries as batter/bowler/
# non_striker/player_out.  (Officials appear in registry but NOT here,
# so this filter drops them from dim_player automatically.)
# ─────────────────────────────────────────────────────────────
d = spark.table(SILVER_DELIVERIES)

participant_names = (
    d.select("match_id", F.col("batter").alias("name")).where("batter is not null")
    .union(d.select("match_id", F.col("bowler").alias("name")).where("bowler is not null"))
    .union(d.select("match_id", F.col("non_striker").alias("name")).where("non_striker is not null"))
    .union(d.select("match_id", F.col("player_out").alias("name")).where("player_out is not null"))
    .distinct()
)

reg = spark.table(SILVER_REGISTRY)

# resolve each participant (match_id, name) -> person_id via THAT match's registry
participants = (participant_names.alias("p")
    .join(reg.alias("r"), ["match_id", "name"], "inner")
    .select("person_id", "name", "match_id")
)

# join match start_date to get first/last seen
dm = spark.table(f"{CATALOG}.silver.dim_match").select("match_id", "start_date", "gender")
part_dated = participants.join(dm, "match_id", "left")

In [0]:
# ─────────────────────────────────────────────────────────────
# player_name bridge: person_id x name spelling, with usage + span
# ─────────────────────────────────────────────────────────────
player_name = (part_dated
    .groupBy("person_id", "name")
    .agg(F.count("*").alias("usage_count"),
         F.min("start_date").alias("first_seen"),
         F.max("start_date").alias("last_seen"))
)
player_name.write.format("delta").mode("overwrite").clusterBy("person_id").saveAsTable(SILVER_PLAYER_NAME)

# ─────────────────────────────────────────────────────────────
# dim_player: canonical_name = most-used spelling (tie-break: most recent)
# ─────────────────────────────────────────────────────────────
from pyspark.sql.window import Window
w = Window.partitionBy("person_id").orderBy(F.col("usage_count").desc(), F.col("last_seen").desc())

canonical = (player_name
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .select("person_id", F.col("name").alias("canonical_name")))

dim_player = (part_dated
    .groupBy("person_id")
    .agg(F.min("start_date").alias("first_match_date"),
         F.max("start_date").alias("last_match_date"),
         F.max("gender").alias("gender"))
    .join(canonical, "person_id", "left")
    .select("person_id", "canonical_name", "gender",
            "first_match_date", "last_match_date"))

dim_player.write.format("delta").mode("overwrite").clusterBy("person_id").saveAsTable(SILVER_DIM_PLAYER)

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify — resolution completeness + officials correctly excluded
# ─────────────────────────────────────────────────────────────
spark.sql(f"""
  SELECT
    (SELECT count(*) FROM {SILVER_DIM_PLAYER})                                   AS players,
    (SELECT count(*) FROM {SILVER_REGISTRY})                                     AS registry_rows,
    (SELECT count(*) FROM {SILVER_DELIVERIES} WHERE batter IS NOT NULL
                                                AND batter_id IS NULL)           AS unresolved_batters,
    (SELECT count(*) FROM {SILVER_PLAYER_NAME}
       WHERE person_id IN (SELECT person_id FROM {SILVER_PLAYER_NAME}
                           GROUP BY person_id HAVING count(*) > 1))              AS ids_with_multiple_names
""").show()

In [0]:
# ─────────────────────────────────────────────────────────────
# Empty curated seeds (schema only) — canonical/enrichment resolve
# to null via left join until you hand-populate them later.
# ─────────────────────────────────────────────────────────────
CATALOG = "cricket"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.silver.team_alias (
    team_name STRING, franchise_key BIGINT,
    franchise_current_name STRING, valid_from DATE, valid_to DATE)""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.silver.venue_alias (
    venue_name STRING, canonical_venue_name STRING)""")

In [0]:
# ─────────────────────────────────────────────────────────────
# dim_team  (grain: 1 distinct team name)
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = f"{CATALOG}.bronze.raw_matches"
dm = spark.table(f"{CATALOG}.silver.dim_match")

# team_type is source-level (not on dim_match) -> pull from bronze
team_type_src = spark.table(BRONZE).select(
    "match_id", F.expr("data:info:team_type::string").alias("team_type"))

teams_long = (dm.select("match_id",
                        F.explode(F.array("team_a", "team_b")).alias("team_name"),
                        "start_date")
                .join(team_type_src, "match_id", "left"))

# predominant team_type per team (mode)
tt_w = Window.partitionBy("team_name").orderBy(F.col("cnt").desc())
team_type_mode = (teams_long.groupBy("team_name", "team_type").agg(F.count("*").alias("cnt"))
                  .withColumn("_rn", F.row_number().over(tt_w)).filter("_rn=1")
                  .select("team_name", "team_type"))

dim_team = (teams_long.groupBy("team_name")
            .agg(F.min("start_date").alias("first_match_date"),
                 F.max("start_date").alias("last_match_date"))
            .join(team_type_mode, "team_name", "left")
            .withColumn("team_key", F.xxhash64(F.lower(F.trim("team_name"))))
            # curated franchise unification (null until team_alias seeded)
            .join(spark.table(f"{CATALOG}.silver.team_alias")
                    .select("team_name", "franchise_key", "franchise_current_name"),
                  "team_name", "left")
            .select("team_key", "team_name", "team_type",
                    "franchise_key", "franchise_current_name",
                    "first_match_date", "last_match_date"))

dim_team.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("team_key").saveAsTable(f"{CATALOG}.silver.dim_team")

In [0]:
# dim_series  (grain: event x season)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
dm = spark.table(f"{CATALOG}.silver.dim_match")

sw = Window.partitionBy("event_name", "season").orderBy(F.col("cnt").desc())
series_fmt = (dm
    .withColumn("event_name", F.coalesce("event_name", F.lit("(no event)")))
    .groupBy("event_name", "season", "match_format").agg(F.count("*").alias("cnt"))
    .withColumn("_rn", F.row_number().over(sw)).filter("_rn=1")
    .select("event_name", "season", "match_format"))

dim_series = (dm
    .withColumn("event_name", F.coalesce("event_name", F.lit("(no event)")))
    .groupBy("event_name", "season")
    .agg(F.min("start_date").alias("first_match_date"),
         F.max("start_date").alias("last_match_date"))
    .join(series_fmt, ["event_name", "season"], "left")
    .withColumn("season_start_year",
                F.regexp_extract("season", r"^(\d{4})", 1).cast("int"))
    .withColumn("series_key",
                F.xxhash64(F.concat_ws("|", "event_name", "season")))
    .select("series_key", "event_name", "season", "season_start_year",
            "match_format", "first_match_date", "last_match_date"))

dim_series.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("series_key").saveAsTable(f"{CATALOG}.silver.dim_series")

In [0]:
# ─────────────────────────────────────────────────────────────
# dim_calendar  (grain: 1 date — contiguous spine for BI time-intel)
# ─────────────────────────────────────────────────────────────
row = dm.agg(F.min("start_date").alias("mn"), F.max("end_date").alias("mx")).collect()[0]
mn, mx = row["mn"], row["mx"]

dim_calendar = (spark.sql(f"""
    SELECT explode(sequence(to_date('{mn}'), to_date('{mx}'), interval 1 day)) AS date
""")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("iso_week", F.weekofyear("date"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))  # Sun=1, Sat=7
    .select("date_key", "date", "year", "month", "day",
            "month_name", "day_name", "quarter", "iso_week", "is_weekend"))

dim_calendar.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.silver.dim_calendar")

In [0]:
# ─────────────────────────────────────────────────────────────
# dim_venue  (grain: 1 distinct venue string; canonical via seed)
# ─────────────────────────────────────────────────────────────
dim_venue = (dm.groupBy("venue", "city")
    .agg(F.min("start_date").alias("first_match_date"),
         F.max("start_date").alias("last_match_date"))
    .withColumn("venue_key", F.xxhash64(F.lower(F.trim("venue"))))
    .join(spark.table(f"{CATALOG}.silver.venue_alias"), "venue", "left")
    .withColumn("canonical_venue_key",
                F.when(F.col("canonical_venue_name").isNotNull(),
                       F.xxhash64(F.lower(F.trim("canonical_venue_name")))))
    .select("venue_key", F.col("venue").alias("venue_name"), "city",
            "canonical_venue_key", "canonical_venue_name",
            "first_match_date", "last_match_date"))

dim_venue.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("venue_key").saveAsTable(f"{CATALOG}.silver.dim_venue")

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify all four
# ─────────────────────────────────────────────────────────────
for t in ["dim_team", "dim_series", "dim_calendar", "dim_venue"]:
    print(t, spark.table(f"{CATALOG}.silver.{t}").count())

spark.sql(f"SELECT team_name, team_type, first_match_date FROM {CATALOG}.silver.dim_team ORDER BY team_name LIMIT 8").show(truncate=False)
spark.sql(f"SELECT event_name, season, match_format FROM {CATALOG}.silver.dim_series ORDER BY first_match_date LIMIT 8").show(truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────
# dim_phase  (small static lookup) + ball_phase (per-ball assignment)
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, ArrayType, DoubleType)

CATALOG = "cricket"
BRONZE  = f"{CATALOG}.bronze.raw_matches"

# --- 1) static dim_phase members ---
phase_rows = [
    ("PP_MANDATORY", "Powerplay (mandatory)", True,  "mandatory"),
    ("PP_BATTING",   "Powerplay (batting)",   True,  "batting"),
    ("PP_FIELDING",  "Powerplay (fielding)",  True,  "fielding"),
    ("NON_PP",       "Non-powerplay",         False, None),
    ("NA",           "N/A (no powerplays)",   False, None),
]
dim_phase = spark.createDataFrame(
    phase_rows, ["phase_key", "phase_name", "is_powerplay", "powerplay_type"])
dim_phase.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{CATALOG}.silver.dim_phase")

In [0]:
# --- 2) parse powerplays per innings from bronze ---
# powerplays: array<struct<from double, to double, type string>>
pp_schema = ArrayType(StructType([
    StructField("from", DoubleType()),
    StructField("to",   DoubleType()),
    StructField("type", StringType()),
]))
inn_pp_schema = ArrayType(StructType([
    StructField("team", StringType()),
    StructField("powerplays", pp_schema),
]))

pp = (spark.table(BRONZE)
    .withColumn("inn", F.from_json(F.expr("to_json(data:innings)"), inn_pp_schema))
    .select("match_id",
            F.posexplode_outer("inn").alias("inn_idx", "innings"))
    .withColumn("innings_number", F.col("inn_idx") + 1)
    .select("match_id", "innings_number",
            F.explode_outer("innings.powerplays").alias("pp"))
    # split cricket-notation "5.6" into over=5, ball=6 (NOT a float!)
    .withColumn("from_over", F.floor("pp.from").cast("int"))
    .withColumn("from_ball", F.round((F.col("pp.from") - F.floor("pp.from")) * 10).cast("int"))
    .withColumn("to_over",   F.floor("pp.to").cast("int"))
    .withColumn("to_ball",   F.round((F.col("pp.to")   - F.floor("pp.to"))   * 10).cast("int"))
    .withColumn("pp_type",   F.col("pp.type"))
    .where("pp is not null")
)

In [0]:
# --- 3) assign each ball a phase via tuple comparison ---
d = spark.table(f"{CATALOG}.silver.deliveries").select(
    "match_id", "innings_number", "over_number", "ball_seq")

# a ball at (over_number, ball_seq) falls in a powerplay when
#   (from_over,from_ball) <= (over,ball) <= (to_over,to_ball)
joined = (d.join(pp, ["match_id", "innings_number"], "left")
    .withColumn("in_pp",
        (F.col("pp_type").isNotNull()) &
        # >= from
        ((F.col("over_number") > F.col("from_over")) |
         ((F.col("over_number") == F.col("from_over")) & (F.col("ball_seq") >= F.col("from_ball")))) &
        # <= to
        ((F.col("over_number") < F.col("to_over")) |
         ((F.col("over_number") == F.col("to_over")) & (F.col("ball_seq") <= F.col("to_ball"))))
    ))

# collapse to one phase per ball: a matching PP wins; else NON_PP; Tests -> NA
ball_phase = (joined
    .withColumn("phase_key",
        F.when(F.col("in_pp") & (F.col("pp_type") == "mandatory"), "PP_MANDATORY")
         .when(F.col("in_pp") & (F.col("pp_type") == "batting"),   "PP_BATTING")
         .when(F.col("in_pp") & (F.col("pp_type") == "fielding"),  "PP_FIELDING"))
    .groupBy("match_id", "innings_number", "over_number", "ball_seq")
    .agg(F.max("phase_key").alias("pp_hit"),           # any PP membership for this ball
         F.max(F.col("pp_type").isNotNull().cast("int")).alias("had_pp"))
    .withColumn("phase_key",
        F.when(F.col("pp_hit").isNotNull(), F.col("pp_hit"))
         .when(F.col("had_pp") == 1, F.lit("NON_PP"))   # match has PPs, ball outside them
         .otherwise(F.lit("NA")))                        # match has no PPs at all (Tests)
    .select("match_id", "innings_number", "over_number", "ball_seq", "phase_key"))

ball_phase.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.silver.ball_phase")

In [0]:
# --- verify ---
spark.sql(f"""
  SELECT bp.phase_key, p.phase_name, count(*) balls
  FROM {CATALOG}.silver.ball_phase bp
  JOIN {CATALOG}.silver.dim_phase  p USING (phase_key)
  GROUP BY bp.phase_key, p.phase_name
  ORDER BY balls DESC
""").show(truncate=False)

# sanity: every delivery got exactly one phase (no balls lost/duplicated)
print("deliveries:", spark.table(f"{CATALOG}.silver.deliveries").count(),
      "| ball_phase:", spark.table(f"{CATALOG}.silver.ball_phase").count())

In [0]:
# ─────────────────────────────────────────────────────────────
# SILVER BRIDGES — wicket, wicket_fielder, match_player, dim_innings
# All re-parse bronze; person names resolved via match_registry.
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               BooleanType, ArrayType, DoubleType, MapType)

CATALOG = "cricket"
BRONZE  = f"{CATALOG}.bronze.raw_matches"
reg     = spark.table(f"{CATALOG}.silver.match_registry")

def resolve(df, name_col, id_col):
    r = reg.select("match_id", F.col("name").alias(name_col),
                   F.col("person_id").alias(id_col))
    return df.join(r, ["match_id", name_col], "left")

In [0]:
# ─────────────────────────────────────────────────────────────
# wicket + wicket_fielder  (re-explode innings->overs->deliveries->wickets->fielders)
# ─────────────────────────────────────────────────────────────
fielder_schema = StructType([StructField("name", StringType()),
                             StructField("substitute", BooleanType())])
wicket_schema  = StructType([StructField("player_out", StringType()),
                             StructField("kind", StringType()),
                             StructField("fielders", ArrayType(fielder_schema))])
delivery_schema = StructType([
    StructField("wickets", ArrayType(wicket_schema))])
over_schema = StructType([StructField("over", IntegerType()),
                          StructField("deliveries", ArrayType(delivery_schema))])
innings_schema = ArrayType(StructType([
    StructField("team", StringType()),
    StructField("overs", ArrayType(over_schema))]))

parsed = (spark.table(BRONZE)
    .withColumn("innings", F.from_json(F.expr("to_json(data:innings)"), innings_schema))
    .select("match_id", F.posexplode_outer("innings").alias("inn_idx", "inn"))
    .withColumn("innings_number", F.col("inn_idx") + 1)
    .select("match_id", "innings_number",
            F.explode_outer("inn.overs").alias("over"))
    .withColumn("over_number", F.col("over.over"))
    .select("match_id", "innings_number", "over_number",
            F.posexplode_outer("over.deliveries").alias("ball_idx", "d"))
    .withColumn("ball_seq", F.col("ball_idx") + 1)
    .where("d is not null"))

# one row per wicket entry (posexplode -> wicket_index handles 2-on-a-ball)
wkt = (parsed
    .select("match_id", "innings_number", "over_number", "ball_seq",
            F.posexplode_outer("d.wickets").alias("wicket_index", "w"))
    .where("w is not null")
    .withColumn("wicket_id",
        F.xxhash64(F.concat_ws("|", "match_id", "innings_number",
                               "over_number", "ball_seq", "wicket_index")))
    .withColumn("kind", F.col("w.kind"))
    .withColumn("player_out", F.col("w.player_out"))
    # bowler-credited set
    .withColumn("is_bowler_wicket",
        F.col("w.kind").isin("bowled","caught","caught and bowled",
                             "lbw","stumped","hit wicket")))

wkt_resolved = resolve(wkt, "player_out", "player_out_id").select(
    "wicket_id", "match_id", "innings_number", "over_number", "ball_seq",
    "wicket_index", "kind", "player_out", "player_out_id", "is_bowler_wicket")

wkt_resolved.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.silver.wicket")

# wicket_fielder: explode fielders under each wicket (multi-fielder run-outs)
wf = (parsed
    .select("match_id", "innings_number", "over_number", "ball_seq",
            F.posexplode_outer("d.wickets").alias("wicket_index", "w"))
    .where("w is not null")
    .withColumn("wicket_id",
        F.xxhash64(F.concat_ws("|", "match_id", "innings_number",
                               "over_number", "ball_seq", "wicket_index")))
    .select("wicket_id", "match_id",
            F.posexplode_outer("w.fielders").alias("fielder_seq", "f"))
    .where("f is not null")
    .withColumn("fielder_name", F.col("f.name"))
    .withColumn("is_substitute", F.coalesce(F.col("f.substitute"), F.lit(False))))

wf_resolved = resolve(wf, "fielder_name", "fielder_id").select(
    "wicket_id", "fielder_seq", "fielder_name", "fielder_id", "is_substitute")

wf_resolved.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("wicket_id").saveAsTable(f"{CATALOG}.silver.wicket_fielder")

In [0]:
# ─────────────────────────────────────────────────────────────
# match_player  (grain: match x team x player — the squad / matches-played)
# players: {team -> [name, ...]}  (12+ with impact subs)
# ─────────────────────────────────────────────────────────────
players_map = (spark.table(BRONZE)
    .withColumn("players",
        F.from_json(F.expr("to_json(data:info:players)"),
                    MapType(StringType(), ArrayType(StringType()))))
    .select("match_id", F.explode("players").alias("team", "names"))
    .select("match_id", "team", F.explode("names").alias("name")))

match_player = resolve(players_map, "name", "person_id").select(
    "match_id", "team", F.col("name").alias("player_name"), "person_id")

match_player.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.silver.match_player")

In [0]:
# ─────────────────────────────────────────────────────────────
# dim_innings  (grain: match x innings — innings-level attributes)
# ─────────────────────────────────────────────────────────────
target_schema = StructType([StructField("overs", DoubleType()),
                            StructField("runs", IntegerType())])
penalty_schema = StructType([StructField("pre", IntegerType()),
                             StructField("post", IntegerType())])
inn_full = ArrayType(StructType([
    StructField("team", StringType()),
    StructField("super_over", BooleanType()),
    StructField("declared", BooleanType()),
    StructField("forfeited", BooleanType()),
    StructField("penalty_runs", penalty_schema),
    StructField("target", target_schema)]))

dim_innings = (spark.table(BRONZE)
    .withColumn("inn", F.from_json(F.expr("to_json(data:innings)"), inn_full))
    .select("match_id", F.posexplode_outer("inn").alias("inn_idx", "x"))
    .withColumn("innings_number", F.col("inn_idx") + 1)
    .select("match_id", "innings_number",
            F.col("x.team").alias("batting_team"),
            F.coalesce(F.col("x.super_over"), F.lit(False)).alias("is_super_over"),
            F.coalesce(F.col("x.declared"),   F.lit(False)).alias("declared"),
            F.coalesce(F.col("x.forfeited"),  F.lit(False)).alias("forfeited"),
            F.coalesce(F.col("x.penalty_runs.pre"),  F.lit(0)).alias("penalty_pre"),
            F.coalesce(F.col("x.penalty_runs.post"), F.lit(0)).alias("penalty_post"),
            F.col("x.target.runs").alias("target_runs"),
            F.col("x.target.overs").alias("target_overs"))
    .where("batting_team is not null"))

dim_innings.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.silver.dim_innings")

In [0]:
CATALOG = "cricket"

# curated exclusions: matches voided from the record (not ordinary abandonments)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.excluded_match (
    match_id STRING,
    reason   STRING,
    excluded_at TIMESTAMP
)""")

spark.sql(f"""
INSERT INTO {CATALOG}.silver.excluded_match
SELECT '1473495', 'Voided and replayed (May 2025 IPL suspension) — replay is the match of record; void excluded from all records', current_timestamp()
WHERE NOT EXISTS (SELECT 1 FROM {CATALOG}.silver.excluded_match WHERE match_id='1473495')
""")

spark.table(f"{CATALOG}.silver.excluded_match").show(truncate=False)

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify
# ─────────────────────────────────────────────────────────────
for t in ["wicket","wicket_fielder","match_player","dim_innings"]:
    print(f"{t:16}", spark.table(f"{CATALOG}.silver.{t}").count())

# wicket count should match is_wicket rows in deliveries (single-wicket balls)
print("deliveries is_wicket:",
      spark.table(f"{CATALOG}.silver.deliveries").where("is_wicket").count())

# fielding-credit sanity: catches per fielder (top 5)
spark.sql(f"""
  SELECT wf.fielder_name, count(*) AS dismissals_involved
  FROM {CATALOG}.silver.wicket_fielder wf
  GROUP BY wf.fielder_name ORDER BY dismissals_involved DESC LIMIT 5
""").show(truncate=False)

# matches-vs-innings check: a player's matches (match_player) vs batting innings (deliveries)
print("match_player rows:", spark.table(f"{CATALOG}.silver.match_player").count())